# Cortex segmenté par WMH-SynthSeg

Le cortex (labels 3 et 42 : gauche et droit) de la segmentation complète de WMH-SynthSeg, sur la grille de la FLAIR, en surimpression sur la FLAIR N4 : un sujet par scanner, 3 coupes axiales. Ce masque sert à la substance grise des indicateurs de qualité (4.1). À regarder : le cortex suit-il le ruban gris, déborde-t-il sur le liquide des sillons ou sur la substance blanche ?

In [ ]:
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

from wmh_multisite.config import load_config, resolve_path

SUBJECTS = None          # None: first subject of each scanner; or a list, e.g. ["sub-000", "sub-132"]
CORTEX = (3, 42)         # left / right cortex in WMH-SynthSeg

cfg = load_config()
D = resolve_path(cfg, "derivatives") / "wmh-multisite"
part = pd.read_csv(resolve_path(cfg, "bids") / "participants.tsv", sep="\t")
if SUBJECTS is None:
    SUBJECTS = part.groupby("scanner").participant_id.first().tolist()
print(SUBJECTS)

In [ ]:
def canon(path):
    return nib.as_closest_canonical(nib.load(str(path)))   # RAS: axis 2 = foot -> head


fig, axes = plt.subplots(len(SUBJECTS), 3, figsize=(12, 4 * len(SUBJECTS)), squeeze=False)
for row, sub in zip(axes, SUBJECTS):
    a = D / sub / "anat"
    img = canon(a / f"{sub}_desc-n4_FLAIR.nii.gz")
    flair = img.get_fdata()
    labels = canon(a / f"{sub}_space-FLAIR_desc-wmhsynthsegfull_dseg.nii.gz").get_fdata()
    cortex = np.isin(labels, CORTEX)
    zooms = img.header.get_zooms()
    ks = np.where(cortex.any(axis=(0, 1)))[0]
    vmax = np.percentile(flair[flair > 0], 99.5)
    scanner = part.set_index("participant_id").scanner[sub]
    for ax, q in zip(row, (0.35, 0.55, 0.75)):
        k = int(np.quantile(ks, q))
        ax.imshow(flair[:, :, k].T, cmap="gray", origin="lower", vmin=0, vmax=vmax, aspect=zooms[1] / zooms[0])
        ax.imshow(np.ma.masked_where(~cortex[:, :, k].T, cortex[:, :, k].T), cmap="autumn", origin="lower",
                  alpha=0.45, aspect=zooms[1] / zooms[0])
        ax.set_title(f"{sub} ({scanner}) - coupe {k}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()